# 099 — Búsqueda léxica y BM25

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** `IDF(df=1) = ln(1 + 99.5/1.5) ≈ ln(67.3) ≈ 4.21`;
`IDF(df=10) = ln(1 + 90.5/10.5) ≈ ln(9.62) ≈ 2.26`;
`IDF(df=90) = ln(1 + 10.5/90.5) ≈ ln(1.116) ≈ 0.11`. Patrón: decaimiento
aproximadamente logarítmico — un término casi omnipresente aporta ~40× menos que uno raro.

**Ejercicio 2.** `N = 3`; `df(red) = 3 → IDF(red) = ln(1 + 0.5/3.5) ≈ 0.134`;
`df(profunda) = 1 → IDF(profunda) = ln(1 + 2.5/1.5) ≈ 0.981`. `avgdl = 5`.

- D1 (3 tok, f(red)=1, f(profunda)=1): factor `= 1.2·(0.25+0.75·0.6) = 0.84`;
  red: `0.134·2.2/1.84 ≈ 0.160`; profunda: `0.981·2.2/1.84 ≈ 1.173` → **1.333**.
- D2 (6 tok, f(red)=2): factor `= 1.2·(0.25+0.75·1.2) = 1.38`;
  red: `0.134·(2·2.2)/(2+1.38) ≈ 0.174` → **0.174**.
- D3 (6 tok, f(red)=1): red: `0.134·2.2/(1+1.38) ≈ 0.124` → **0.124**
  ("profundo" ≠ "profunda" sin stemming — el vocabulary mismatch en acción).

Ranking: **D1 > D2 > D3**. "red" aparece en los 3 documentos y su IDF casi lo anula;
el término raro "profunda" decide el ranking.

**Ejercicio 3.** Con `k₁ = 0.1` (factor `= 0.1·1.4/1.2… ≈ 0.115`):
red en D2 con f=2 da ≈ `0.139` frente a `0.132` con f=1 — casi binario:
la segunda aparición no añade prácticamente nada. Con `k₁ = 10` (factor `= 11.5`):
`0.134·(2·11)/(2+11.5) ≈ 0.218` frente a `0.118` con f=1 — casi lineal: cada aparición
sigue sumando. `k₁` interpola entre "presencia" y "conteo crudo".

**Ejercicio 4.** El contrato (`kind`, `evidence`) es estable; los valores dependen de la semilla.


In [ ]:
result = run_lab("retrieval", seed=99)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
import math

def idf(N, df):
    return math.log(1 + (N - df + 0.5) / (df + 0.5))

def bm25_term(f, dl, avgdl, N, df, k1=1.2, b=0.75):
    if f == 0:
        return 0.0
    return idf(N, df) * f * (k1 + 1) / (f + k1 * (1 - b + b * dl / avgdl))

print("E1:", [round(idf(100, d), 2) for d in (1, 10, 90)])

# Ejercicio 2 — Q = {red, profunda}, avgdl = 5, N = 3
docs = {  # nombre: (longitud, f_red, f_profunda)
    "D1": (3, 1, 1),
    "D2": (6, 2, 0),
    "D3": (6, 1, 0),
}
for name, (dl, fr, fp) in docs.items():
    s = bm25_term(fr, dl, 5, 3, 3) + bm25_term(fp, dl, 5, 3, 1)
    print(name, "->", round(s, 3))

# Ejercicio 3 — saturación en D2
for k1 in (0.1, 1.2, 10):
    s1 = bm25_term(1, 6, 5, 3, 3, k1=k1)
    s2 = bm25_term(2, 6, 5, 3, 3, k1=k1)
    print(f"k1={k1}: f=1 -> {s1:.3f}  f=2 -> {s2:.3f}  ganancia={s2/s1:.2f}x")


## Reflexión

1. En el ejemplo, D2 repite "gato" y aun así pierde contra D1. ¿Qué dos componentes de la fórmula producen ese resultado y qué pasaría con k₁ = 100?
2. ¿Por qué los scores BM25 de dos consultas distintas no son comparables entre sí, y qué problema causa eso al fusionar rankings (anticipa la clase 100)?
3. Da dos ejemplos de consultas de tu dominio donde BM25 debería ganar a los embeddings, y dos donde debería perder. ¿Cómo lo comprobarías?
